In [1]:
import os
import re
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, SnowballStemmer, PorterStemmer
from nltk.corpus import stopwords
import spacy
from matplotlib import pyplot as plt
from sklearn.feature_extraction.text import TfidfTransformer, CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from tqdm import tqdm

from scipy.sparse import csr_matrix
from gensim import corpora, models
from gensim.matutils import Sparse2Corpus

from scipy.stats import pearsonr

In [2]:
# Function to download NLTK resources
def download_nltk_resources():
    required_resources = ['wordnet', 'stopwords', 'punkt', 'punkt_tab']
    for resource in required_resources:
        try:
            nltk.data.find(f'tokenizers/{resource}' if resource == 'punkt' else f'corpora/{resource}')
        except LookupError:
            nltk.download(resource)

download_nltk_resources()

# Load spaCy model; if missing, install it in the current Jupyter kernel environment.
try:
    sp = spacy.load('en_core_web_sm')
except OSError:
    from spacy.cli import download as spacy_download

    print("Installing spaCy model 'en_core_web_sm' for this environment...")
    spacy_download('en_core_web_sm')
    sp = spacy.load('en_core_web_sm')

# Enable tqdm for pandas
tqdm.pandas()

# Initialize stemmers and lemmatizer
porter = SnowballStemmer("english")
lmtzr = WordNetLemmatizer()
STOP_WORDS = set(stopwords.words('english'))


[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/andregarcia/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/andregarcia/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
"""
This module provides helper functions for text preprocessing. 
Each function applies punctuation removal and stopword removal, and then one of three options:
    0: Lowercasing only.
    1: Lowercasing plus stemming.
    2: Lemmatizing (using spaCy; original casing is preserved).

The functions return a string of tokens separated by spaces.
"""

def preprocess_lower(text):
    """
    Preprocess text by:
       - Converting to lowercase.
       - Removing punctuation.
       - Tokenizing.
       - Removing stopwords.
    
    Returns:
        str: A string of filtered tokens separated by spaces.
    """
    text_lower = text.lower()
    text_no_punct = re.sub(r'[^\w\s]', '', text_lower)
    tokens = word_tokenize(text_no_punct)
    filtered_tokens = [token for token in tokens if token not in STOP_WORDS]
    return " ".join(filtered_tokens)

def preprocess_stem(text):
    """
    Preprocess text by performing all steps in preprocess_lower() and then applying stemming.
    
    Returns:
        str: A string of stemmed tokens separated by spaces.
    """
    tokens = preprocess_lower(text).split()
    ps = PorterStemmer()
    stemmed_tokens = [ps.stem(token) for token in tokens]
    return " ".join(stemmed_tokens)

def preprocess_lemma(text):
    """
    Preprocess text by:
       - Removing punctuation and stopwords using spaCy's token attributes.
       - Lemmatizing the text.
       - (Note: This function does NOT lowercase the text.)
    
    Returns:
        str: A string of lemmatized tokens separated by spaces.
    """
    doc = sp(text)
    lemmatized_tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct and token.lemma_.strip() != '']
    return " ".join(lemmatized_tokens)

def tokenize(text, mode=0):
    """
    General tokenize function. Always applies punctuation and stopword removal and then:
    
      mode = 0: Applies lowercasing.
      mode = 1: Applies lowercasing and stemming.
      mode = 2: Applies lemmatization (without lowercasing the original text).
    
    Args:
        text (str): The input text to be processed.
        mode (int): Processing mode (0 for lowercasing; 1 for stemming; 2 for lemmatizing).

    Returns:
        str: A string of processed tokens separated by spaces.

    Raises:
        ValueError: If an invalid mode is provided.
    """
    if mode == 0:
        return preprocess_lower(text)
    elif mode == 1:
        return preprocess_stem(text)
    elif mode == 2:
        return preprocess_lemma(text)
    else:
        raise ValueError("Invalid mode. Please use 0 for lowercasing, 1 for stemming, or 2 for lemmatizing.")


In [4]:
data_file = 'hansard.csv'
corpus_data = pd.read_csv(data_file, encoding='utf-8')

corpus_data = corpus_data.dropna(subset='speaker')
corpus_data.drop(columns=["file","speech_id","type","speaker_id", "colnum","time","url"], 
                 inplace=True)

corpus_data.info()
corpus_data.head()

/var/folders/1g/np2l7mrj4zd9sdx0dkvqh4rh0000gn/T/ipykernel_17719/1378335376.py:2: DtypeWarning: Columns (3,5,6,8) have mixed types. Specify dtype option on import or set low_memory=False.
  corpus_data = pd.read_csv(data_file, encoding='utf-8')


<class 'pandas.core.frame.DataFrame'>
Index: 4328210 entries, 5 to 4671401
Data columns (total 5 columns):
 #   Column      Dtype 
---  ------      ----- 
 0   date        object
 1   speaker     object
 2   section     object
 3   text        object
 4   word_count  int64 
dtypes: int64(1), object(4)
memory usage: 198.1+ MB


,date,speaker,section,text,word_count
5,1952-01-29,Mr. Stephen Swingler,Re-armament and Essential Industries,asked the Minister of Labour whether he will m...,20
6,1952-01-29,Mr. Norman Dodds,Re-armament and Essential Industries,asked the Minister of Labour what proposals he...,26
7,1952-01-29,The Minister of Labour (Sir Walter Monckton),Re-armament and Essential Industries,I would ask the hon. Members to await the stat...,34
8,1952-01-29,Mr. Swingler,Re-armament and Essential Industries,asked the Minister of Labour what numbers of p...,28
9,1952-01-29,Sir W. Monckton,Re-armament and Essential Industries,I regret that it is not practicable to make su...,12


In [ ]:
data_file = 'UK_monarch_speeches.csv'
monarch_data = pd.read_csv(data_file, encoding='utf-8')

monarch_data.drop(columns=["year","title","url"], 
                 inplace=True)
monarch_data.rename(columns={"monarch": "speaker", "speech": "text"}, inplace=True)
monarch_data['section'] = 'Monarch speech identifier'
monarch_data["word_count"] = monarch_data["text"].apply(lambda x: len(x.split()))

monarch_data.info()
monarch_data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 433 entries, 0 to 432
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   date        433 non-null    object
 1   speaker     433 non-null    object
 2   text        433 non-null    object
 3   section     433 non-null    object
 4   word_count  433 non-null    int64 
dtypes: int64(1), object(4)
memory usage: 17.0+ KB


,date,speaker,text,section,word_count
0,1952-12-25,Queen Elizabeth II,"Each Christmas, at this time, my beloved fathe...",Monarch speech identifier,690
1,1953-06-02,Queen Elizabeth II,"When I spoke to you last, at Christmas, I aske...",Monarch speech identifier,641
2,1953-06-02,Queen Elizabeth II,"In the Coronation ceremony of 2 June 1953, one...",Monarch speech identifier,451
3,1953-12-25,Queen Elizabeth II,Last Christmas I spoke to you from England; th...,Monarch speech identifier,900
4,1954-12-25,Queen Elizabeth II,It is now two years since my husband and I spe...,Monarch speech identifier,641


In [ ]:
corpus_data = pd.concat([corpus_data, monarch_data], ignore_index=True)
corpus_data.tail()

,date,speaker,section,text,word_count
4328638,2025-12-12,King Charles III,Monarch speech identifier,This is a season when our thoughts turn to cel...,636
4328639,2025-12-19,King Charles III,Monarch speech identifier,"Ladies and gentlemen, I am delighted to be abl...",886
4328640,2026-02-11,King Charles III,Monarch speech identifier,My wife and I were profoundly shocked and sadd...,365
4328641,2026-02-12,King Charles III,Monarch speech identifier,So to those who provide care - whether you wea...,754
4328642,2026-02-18,King Charles III,Monarch speech identifier,My wife and I were deeply saddened to hear of ...,96


In [7]:
# WATCH OUT this will take time 10 Minutes roughly
# Set mode (0 = lowercasing, 1 = stemming, 2 = lemmatizing)
mod = 1

# Pre-process the text column with progress tracking
try:
    corpus_data["text"] = corpus_data.text.astype(str).progress_apply(lambda row: tokenize(row, mod))
    corpus_data["word_count"] = corpus_data.text.progress_apply(lambda x: len(x.split()))
    print("Done processing text.")
except Exception as e:
    print(f"Error processing text column: {e}")
    raise RuntimeError(f"Error processing text column: {e}") from e

100%|██████████| 4328643/4328643 [00:08<00:00, 496442.95it/s]

Done processing text.


In [8]:
#corpus_data.to_csv("speeches_processed.csv", index=False)
corpus_data = corpus_data[corpus_data['word_count']>20]
corpus_data.reset_index(drop=True, inplace=True)

In [ ]:
cv = CountVectorizer(ngram_range = (1,2), lowercase=True, min_df=0.005, max_df=0.7)
cv.fit(corpus_data["text"])
vectorized_text=cv.transform(corpus_data["text"])
vectorized_text.shape

(2566390, 1057)

In [10]:
#gensim needs the dtm in a different sparce format
dtm_sparse = csr_matrix(vectorized_text)

# Convert sparse matrix to gensim corpus
corpus = Sparse2Corpus(dtm_sparse, documents_columns=False)

# Create gensim dictionary from sklearn CountVectorizer vocabulary
vocabulary_gensim = {}
for key, val in cv.vocabulary_.items():
    vocabulary_gensim[val] = key

dictionary = corpora.Dictionary()
dictionary.id2token = vocabulary_gensim
dictionary.token2id = cv.vocabulary_

In [12]:
# Number of topics
num_topics = 50

# Build LDA model
lda_model = models.LdaModel(corpus=corpus, 
                            id2word=dictionary, 
                            num_topics=num_topics, 
                            passes=20, 
                            random_state=123)

In [13]:
# Display the topics
topics = lda_model.print_topics(num_words=50)
for topic in topics:
    print(topic)
    print("    ")

(7, '0.217*"busi" + 0.136*"number" + 0.076*"group" + 0.075*"oper" + 0.067*"small" + 0.042*"account" + 0.040*"inquiri" + 0.038*"control" + 0.035*"failur" + 0.034*"freedom" + 0.033*"damag" + 0.029*"spent" + 0.027*"led" + 0.026*"resourc" + 0.024*"head" + 0.024*"larg" + 0.016*"none" + 0.013*"take" + 0.012*"limit" + 0.010*"occur" + 0.008*"includ" + 0.007*"take account" + 0.006*"exampl" + 0.002*"caus" + 0.001*"greater" + 0.001*"less" + 0.001*"result" + 0.000*"variou" + 0.000*"posit" + 0.000*"presid" + 0.000*"polit" + 0.000*"polici" + 0.000*"primari" + 0.000*"plan" + 0.000*"popul" + 0.000*"price" + 0.000*"post" + 0.000*"previou" + 0.000*"potenti" + 0.000*"power" + 0.000*"practic" + 0.000*"prime minist" + 0.000*"prevent" + 0.000*"precis" + 0.000*"prepar" + 0.000*"pressur" + 0.000*"press" + 0.000*"present" + 0.000*"possibl" + 0.000*"polic"')
    
(48, '0.188*"children" + 0.151*"school" + 0.146*"report" + 0.098*"educ" + 0.090*"committe" + 0.054*"transport" + 0.043*"recommend" + 0.037*"commiss" +

In [14]:
doc_topics = [lda_model.get_document_topics(item, minimum_probability=0.0000001) for item in corpus]

# Initialize the columns for each topic with default value 0
for topic_num in range(num_topics):
    corpus_data[f'topic_{topic_num}'] = 0.0

# Iterate over the document topics to fill the DataFrame
for i, doc_distribution in enumerate(doc_topics):
    for topic_num, prob in doc_distribution:
        corpus_data.at[i, f'topic_{topic_num}'] = prob


In [15]:
corpus_data.head()

,date,speaker,section,text,word_count,topic_0,topic_1,topic_2,topic_3,topic_4,...,topic_40,topic_41,topic_42,topic_43,topic_44,topic_45,topic_46,topic_47,topic_48,topic_49
0,1952-01-29,Mr. Swingler,Re-armament and Essential Industries,view fact minist made number statement speech ...,22,0.048571,0.000952,0.000952,0.000952,0.075310,...,0.000952,0.048881,0.000952,0.000952,0.000952,0.000952,0.000952,0.000952,0.000952,0.000952
1,1952-01-29,Sir W. Monckton,Re-armament and Essential Industries,difficulti make estim chang occup peopl precis...,27,0.292528,0.000833,0.000833,0.000833,0.000833,...,0.000833,0.000833,0.110390,0.000833,0.042503,0.000833,0.000833,0.000833,0.000833,0.000833
2,1952-01-29,Sir W. Monckton,Coalmining (Italian Workers),presum hon member refer italian worker brought...,24,0.200026,0.001053,0.001053,0.001053,0.001053,...,0.001053,0.001053,0.001053,0.001053,0.001053,0.001053,0.001053,0.001053,0.001053,0.001053
3,1952-01-29,Sir W. Monckton,Textiles and Clothing,latest date figur avail 10th decemb date 7379 ...,39,0.221678,0.000833,0.000833,0.000833,0.000833,...,0.042501,0.000833,0.000833,0.000833,0.042502,0.000833,0.000833,0.042816,0.000833,0.000833
4,1952-01-29,Sir W. Monckton,Textiles and Clothing,refer temporari phase due fact earli decemb nu...,43,0.059416,0.029998,0.000588,0.059417,0.000588,...,0.000588,0.000588,0.000588,0.000588,0.000588,0.000588,0.000588,0.000588,0.000588,0.000588


In [16]:
topic_cols = [col for col in corpus_data.columns if col.startswith('topic_')]


daily_sum = corpus_data.assign(
    **{col: corpus_data[col] * corpus_data['word_count'] for col in topic_cols}
).groupby('date')[topic_cols + ['word_count']].sum()


final_daily_vectors = daily_sum[topic_cols].div(daily_sum['word_count'], axis=0).reset_index()

print(final_daily_vectors.head())

: 

In [ ]:
corpus_data['date'] = pd.to_datetime(corpus_data['date'])
corpus_data['year'] = corpus_data['date'].dt.year

/var/folders/1g/np2l7mrj4zd9sdx0dkvqh4rh0000gn/T/ipykernel_7863/2706962720.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  corpus_data['year'] = corpus_data['date'].dt.year


In [ ]:
# 1. Filter out ONLY the Monarch's speeches
royal_speeches = corpus_data[corpus_data['section'] == 'Monarch speech identifier']

# 2. Aggregate them by year using the weighted average method
royal_sum = royal_speeches.assign(
    **{col: royal_speeches[col] * royal_speeches['word_count'] for col in topic_cols}
).groupby('year')[topic_cols + ['word_count']].sum()

royal_vectors = royal_sum[topic_cols].div(royal_sum['word_count'], axis=0).reset_index()

# 3. Rename columns so we don't mix them up with parliament topics later
royal_vectors = royal_vectors.rename(columns={col: f'{col}_r' for col in topic_cols})

royal_vectors.head()

,year,topic_0_r,topic_1_r,topic_2_r,topic_3_r,topic_4_r,topic_5_r,topic_6_r,topic_7_r,topic_8_r,...,topic_90_r,topic_91_r,topic_92_r,topic_93_r,topic_94_r,topic_95_r,topic_96_r,topic_97_r,topic_98_r,topic_99_r
0,1952,0.000043,0.000043,0.010234,0.000043,0.000043,0.000043,0.000043,0.004390,0.000043,...,0.000043,0.000043,0.013447,0.005426,0.000043,0.000043,0.000043,0.018191,0.004806,0.005240
1,1953,0.004183,0.009282,0.006281,0.011938,0.000047,0.000047,0.021178,0.002434,0.000047,...,0.023667,0.000047,0.005665,0.007039,0.012306,0.001832,0.000047,0.009473,0.007463,0.002171
2,1954,0.013394,0.007663,0.007882,0.000057,0.000057,0.006780,0.000057,0.000057,0.000057,...,0.006125,0.005800,0.020935,0.032357,0.024468,0.000057,0.000057,0.000057,0.000057,0.006147
3,1955,0.012236,0.032090,0.000039,0.018086,0.000039,0.000039,0.000039,0.007623,0.000039,...,0.016890,0.000039,0.016115,0.000039,0.024001,0.000039,0.004053,0.004636,0.005334,0.000039
4,1956,0.004123,0.000038,0.025321,0.000038,0.005286,0.000038,0.006804,0.000038,0.004028,...,0.008555,0.000038,0.004049,0.005207,0.006598,0.000038,0.011720,0.006356,0.015345,0.007627


In [ ]:
royal_speeches = corpus_data[corpus_data['section'] == 'Monarch speech identifier']
royal_speeches

,date,speaker,section,text,word_count,topic_0,topic_1,topic_2,topic_3,topic_4,...,topic_91,topic_92,topic_93,topic_94,topic_95,topic_96,topic_97,topic_98,topic_99,year
11758,1952-12-25,Queen Elizabeth II,Monarch speech identifier,Christmas time beloved father broadcast messag...,259,0.000043,0.000043,0.010234,0.000043,0.000043,...,0.000043,0.013447,0.005426,0.000043,0.000043,0.000043,0.018191,0.004806,0.005240,1952
11759,1953-06-02,Queen Elizabeth II,Monarch speech identifier,speak Christmas ask religion pray day Coronati...,227,0.000054,0.025258,0.007809,0.016840,0.000054,...,0.000054,0.000054,0.006431,0.009682,0.000054,0.000054,0.000054,0.006289,0.007455,1953
11760,1953-06-02,Queen Elizabeth II,Monarch speech identifier,Coronation ceremony 2 June 1953 highlight Quee...,205,0.006863,0.007787,0.000065,0.014734,0.000065,...,0.000065,0.000065,0.006572,0.000065,0.000065,0.000065,0.000065,0.015919,0.000065,1953
11761,1953-12-25,Queen Elizabeth II,Monarch speech identifier,Christmas speak England year New Zealand Auckl...,359,0.005264,0.000033,0.008863,0.007242,0.000033,...,0.000033,0.012410,0.007691,0.020956,0.003964,0.000033,0.020800,0.003376,0.000033,1953
11762,1954-12-25,Queen Elizabeth II,Monarch speech identifier,year husband spend Christmas child today look ...,268,0.013394,0.007663,0.007882,0.000057,0.000057,...,0.005800,0.020935,0.032357,0.024468,0.000057,0.000057,0.000057,0.000057,0.006147,1954
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12186,2025-12-12,King Charles III,Monarch speech identifier,season thought turn celebration friend family ...,269,0.000047,0.000047,0.004952,0.026211,0.000047,...,0.000047,0.000047,0.028579,0.020075,0.000047,0.000047,0.000047,0.000047,0.000047,2025
12187,2025-12-19,King Charles III,Monarch speech identifier,lady gentleman delighted able join today Brita...,377,0.000033,0.000033,0.010500,0.026413,0.005941,...,0.000033,0.018918,0.008485,0.022435,0.010956,0.007099,0.024126,0.019386,0.000033,2025
12188,2026-02-11,King Charles III,Monarch speech identifier,wife profoundly shock sadden learn dreadful at...,265,0.000132,0.000132,0.000132,0.000132,0.000132,...,0.000132,0.026450,0.029940,0.000132,0.000132,0.013290,0.000132,0.000132,0.000132,2026
12189,2026-02-12,King Charles III,Monarch speech identifier,provide care wear uniform simply clothe feel a...,318,0.010601,0.000040,0.000040,0.005336,0.000040,...,0.000040,0.000040,0.046475,0.022527,0.000040,0.000040,0.000040,0.018638,0.000040,2026


In [ ]:
# 1. Filter out the Monarch's speeches
parl_speeches = corpus_data[corpus_data['section'] != 'Monarch speech identifier']

# 2. Aggregate daily using the weighted average method
daily_sum = parl_speeches.assign(
    **{col: parl_speeches[col] * parl_speeches['word_count'] for col in topic_cols}
).groupby('date')[topic_cols + ['word_count']].sum()

daily_vectors = daily_sum[topic_cols].div(daily_sum['word_count'], axis=0).reset_index()

# 3. Apply a 3-day rolling window to smooth the noise
rolling_parl = daily_vectors.set_index('date')[topic_cols].rolling('3D', min_periods=1).mean().reset_index()

# Re-add the year column for merging
rolling_parl['year'] = rolling_parl['date'].dt.year

rolling_parl.head()

,date,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,...,topic_91,topic_92,topic_93,topic_94,topic_95,topic_96,topic_97,topic_98,topic_99,year
0,1952-01-29,0.012798,0.009612,0.007904,0.003615,0.005386,0.028327,0.003230,0.004355,0.003665,...,0.001726,0.001330,0.006233,0.003182,0.008795,0.001109,0.002667,0.001951,0.007216,1952
1,1952-01-30,0.009286,0.006895,0.007231,0.004876,0.003860,0.024381,0.004701,0.007354,0.003114,...,0.002702,0.004182,0.005887,0.004661,0.010092,0.002088,0.003551,0.004154,0.004729,1952
2,1952-01-31,0.012137,0.007705,0.006545,0.004729,0.005302,0.029406,0.004857,0.009086,0.003011,...,0.003569,0.005301,0.005411,0.004306,0.007875,0.002746,0.004085,0.004445,0.004257,1952
3,1952-02-01,0.011356,0.008586,0.006897,0.005110,0.005099,0.036839,0.006129,0.009433,0.002488,...,0.005004,0.006395,0.006341,0.008206,0.016046,0.004348,0.004231,0.004132,0.001939,1952
4,1952-02-04,0.004982,0.008760,0.004306,0.005669,0.008164,0.038833,0.001838,0.003230,0.002015,...,0.002998,0.000254,0.006981,0.011354,0.006192,0.002263,0.003613,0.004248,0.007348,1952


In [ ]:
from scipy.spatial.distance import jensenshannon

# 1. Merge the data so each parliamentary day lines up with its corresponding year
final_df = pd.merge(rolling_parl, royal_vectors, on='year', how='left')

# 2. Isolate the two sets of vectors as numpy arrays
p_matrix = final_df[topic_cols].values
r_matrix = final_df[[f'{col}_r' for col in topic_cols]].values

# 3. Calculate Vectorized Jensen-Shannon DISTANCE
# Note: scipy calculates distance across the rows using axis=1
js_distance = jensenshannon(p_matrix, r_matrix, axis=1)

# 4. Convert Distance to SIMILARITY (1 = identical, 0 = completely different)
final_df['similarity_score'] = 1 - js_distance

# View the final result
print(final_df[['date', 'year', 'similarity_score']].tail())


         date  year  similarity_score
95 1952-07-03  1952          0.372413
96 1952-07-04  1952          0.368602
97 1952-07-07  1952          0.379267
98 1952-07-08  1952          0.368978
99 1952-07-09  1952          0.373731


In [ ]:
# 1. Get the exact date of the royal speech for each year
# We take the first occurrence in case the speech spanned multiple rows
speech_dates = royal_speeches.groupby('year')['date'].first().reset_index()
speech_dates = speech_dates.rename(columns={'date': 'speech_date'})

# 2. Merge these speech dates back into your main daily dataframe
final_df = pd.merge(final_df, speech_dates, on='year', how='left')

# 3. Calculate the 'time' variable (days relative to the speech)
final_df['time'] = (final_df['date'] - final_df['speech_date']).dt.days

# 4. Create the 'post' dummy variable (1 if after the speech, 0 if before)
# Note: We strictly use > 0 so the exact day of the speech (0) isn't counted as a "reaction" day
final_df['post'] = (final_df['time'] > 0).astype(int)

# 5. Create the interaction term to measure the change in momentum
final_df['time_x_post'] = final_df['time'] * final_df['post']

# Check your final, regression-ready dataset!
print(final_df[['date', 'speech_date', 'time', 'post', 'time_x_post', 'similarity_score']].head(10))

        date speech_date  time  post  time_x_post  similarity_score
0 1952-01-29  1952-12-25  -331     0            0          0.381869
1 1952-01-30  1952-12-25  -330     0            0          0.391046
2 1952-01-31  1952-12-25  -329     0            0          0.388780
3 1952-02-01  1952-12-25  -328     0            0          0.383543
4 1952-02-04  1952-12-25  -325     0            0          0.357981
5 1952-02-05  1952-12-25  -324     0            0          0.376747
6 1952-02-06  1952-12-25  -323     0            0          0.377184
7 1952-02-07  1952-12-25  -322     0            0          0.384944
8 1952-02-08  1952-12-25  -321     0            0          0.332421
9 1952-02-11  1952-12-25  -318     0            0          0.308998


In [120]:
# 1. Define your window size (e.g., 14 days before and after)
window_days = 14

# 2. Filter the dataframe to strictly keep rows within this time boundary
windowed_df = final_df[
    (final_df['time'] >= -window_days) & 
    (final_df['time'] <= window_days)
].copy()

# 3. BEST PRACTICE: Drop the exact day of the event (time == 0)
# Why? The speech happens during the day. Debates on t=0 are a messy mix of 
# "pre-speech" anticipation and "post-speech" immediate reaction. 
# Dropping it creates a clean mathematical break between the pre and post periods.
windowed_df = windowed_df[windowed_df['time'] != 0]

# 4. Check how many rows you have left for your regression
print(f"Dataframe reduced to {len(windowed_df)} valid parliamentary days within the +/- {window_days} day window.")
print(windowed_df['time'].value_counts().sort_index())

Dataframe reduced to 0 valid parliamentary days within the +/- 14 day window.
Series([], Name: count, dtype: int64)
